# Basic Pinch and `dt_cont` Sensitivity
This notebook shows how the pinch, utility targets, and graph shapes move when the minimum approach temperature assumption changes.


## Case context
The packaged `crude_preheat_train.json` case represents a small crude-unit preheat train with named hot and cold duties. The workflow is: solve the baseline case, inspect the main graphs, then rerun the case after multiplying every stream and utility `dt_cont` value by the same factor.


In [ ]:
from copy import deepcopy
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd

from OpenPinch import PinchProblem
from OpenPinch.resources import copy_sample_case

workspace = TemporaryDirectory()
work_dir = Path(workspace.name)
case_path = copy_sample_case(
    "crude_preheat_train.json",
    work_dir / "crude_preheat_train.json",
)
problem = PinchProblem(problem_filepath=case_path)
summary = problem.summary_frame()
base_row = summary.loc[
    summary["Target"] == "crude_preheat_train/Direct Integration",
    [
        "Hot Utility Target",
        "Cold Utility Target",
        "Heat Recovery",
        "Hot Pinch",
        "Cold Pinch",
    ],
]
base_row


In [ ]:
composite_curve = problem.plot_composite_curve()
grand_composite_curve = problem.plot_grand_composite_curve()
{
    "composite_traces": len(composite_curve.data),
    "grand_composite_traces": len(grand_composite_curve.data),
    "available_graphs": sorted(problem.graph_catalog()["Graph Type"].unique()),
}


In [ ]:
base_payload = json.loads(case_path.read_text(encoding="utf-8"))
multipliers = [0.8, 1.0, 1.2]


def _scale_dt_cont(value, factor: float):
    if value is None:
        return None
    if isinstance(value, dict):
        updated = dict(value)
        if updated.get("value") is not None:
            updated["value"] = float(updated["value"]) * factor
        return updated
    return float(value) * factor


rows = []
for factor in multipliers:
    payload = deepcopy(base_payload)
    for stream in payload["streams"]:
        stream["dt_cont"] = _scale_dt_cont(stream.get("dt_cont"), factor)
    for utility in payload.get("utilities", []):
        utility["dt_cont"] = _scale_dt_cont(utility.get("dt_cont"), factor)

    variant_path = work_dir / f"crude_preheat_train_dt_{factor:.1f}.json"
    variant_path.write_text(json.dumps(payload), encoding="utf-8")

    variant_problem = PinchProblem(problem_filepath=variant_path)
    variant_summary = variant_problem.summary_frame()
    row = variant_summary.loc[
        variant_summary["Target"] == f"{variant_path.stem}/Direct Integration"
    ].iloc[0]
    rows.append(
        {
            "dt_cont multiplier": factor,
            "Hot Utility Target": row["Hot Utility Target"],
            "Cold Utility Target": row["Cold Utility Target"],
            "Heat Recovery": row["Heat Recovery"],
            "Hot Pinch": row["Hot Pinch"],
            "Cold Pinch": row["Cold Pinch"],
        }
    )

pd.DataFrame(rows)


## Inspect these outputs first
Start with the baseline compact summary to confirm the direct-integration target. Then compare the composite and grand composite curves before reading the `dt_cont` sensitivity table. The sensitivity table is the main decision aid: larger `dt_cont` values usually push utility demand up and reduce direct recovery.

## Interpretation
Treat `dt_cont` as a design assumption, not a fixed truth. If the sensitivity table swings sharply across plausible multipliers, the preheat train economics and targeting story are assumption-sensitive and should be discussed explicitly before making retrofit decisions.
